In [7]:
# ============================================================
# GOVERNANCE OFFICER ANALYSIS
# ============================================================
# This notebook performs:
# 1. PII identification
# 2. Pseudonymization demonstration
# 3. GDPR compliance mapping
# 4. EU AI Act analysis
# 5. Governance recommendations
# ============================================================

# Import libraries
import pandas as pd
import hashlib
import numpy as np
from datetime import datetime

print("=" * 60)
print("LOADING CLEANED CREDIT APPLICATIONS DATA")
print("=" * 60)

# Load the cleaned CSV file (created by the Data Engineer)
# The file is located in the data folder (one level up from notebooks)
df = pd.read_csv('../data/credit_applications_clean.csv')

print(f"\n Data loaded successfully!")
print(f"   - Shape: {df.shape}")
print(f"   - Rows: {df.shape[0]}")
print(f"   - Columns: {df.shape[1]}")
print(f"\nFirst 5 rows:")
print(df.head())

print("\n" + "=" * 60)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 60)
print(df.dtypes)

LOADING CLEANED CREDIT APPLICATIONS DATA

 Data loaded successfully!
   - Shape: (501, 20)
   - Rows: 501
   - Columns: 20

First 5 rows:
       _id                                  spending_behavior  \
0  app_200  [{'category': 'Shopping', 'amount': 480}, {'ca...   
1  app_037  [{'category': 'Rent', 'amount': 608}, {'catego...   
2  app_215              [{'category': 'Rent', 'amount': 109}]   
3  app_024           [{'category': 'Fitness', 'amount': 575}]   
4  app_184     [{'category': 'Entertainment', 'amount': 463}]   

        processing_timestamp applicant_info.full_name  \
0  2024-01-15 00:00:00+00:00              Jerry Smith   
1                        NaN           Brandon Walker   
2                        NaN              Scott Moore   
3                        NaN               Thomas Lee   
4  2024-01-15 00:00:00+00:00          Brian Rodriguez   

         applicant_info.email applicant_info.ssn applicant_info.ip_address  \
0   jerry.smith17@hotmail.com        596-64-4340  

## 1. PII Identification (Personal Identifiable Information)

According to GDPR (Article 4), PII is any information that can directly or indirectly identify a natural person.

In the NovaCred dataset, the following PII fields were identified:

| Field | Data Type | Risk Level | Justification |
|-------|-----------|------------|---------------|
| `applicant_info.full_name` | String | High | Direct identification |
| `applicant_info.email` | String | High | Direct identification |
| `applicant_info.ssn` | String | **Critical** | Social Security Number (unique identifier, extremely sensitive) |
| `applicant_info.ip_address` | String | Medium | Can be used to derive location or identity |
| `applicant_info.date_of_birth` | String | Medium | When combined with other data, can identify |

**Note:** The Data Engineer removed columns with >95% missing values, but **sensitive columns (SSN, IP) are still present** in this cleaned version. This represents a privacy risk that needs to be addressed.

In [3]:
print("=" * 60)
print("CHECKING SENSITIVE COLUMNS")
print("=" * 60)

# Check if SSN column exists
if 'applicant_info.ssn' in df.columns:
    print(f"\n Column 'applicant_info.ssn' exists!")
    print(f"   - Number of non-null SSNs: {df['applicant_info.ssn'].notna().sum()}")
    print(f"   - Sample values: {df['applicant_info.ssn'].head(3).tolist()}")
else:
    print("\n Column 'applicant_info.ssn' NOT found (good for privacy)")

# Check if IP address column exists
if 'applicant_info.ip_address' in df.columns:
    print(f"\n Column 'applicant_info.ip_address' exists!")
    print(f"   - Number of non-null IPs: {df['applicant_info.ip_address'].notna().sum()}")
    print(f"   - Sample values: {df['applicant_info.ip_address'].head(3).tolist()}")
else:
    print("\n Column 'applicant_info.ip_address' NOT found (good for privacy)")

# Check if email column exists (also PII)
if 'applicant_info.email' in df.columns:
    print(f"\n Column 'applicant_info.email' exists (PII)")
    print(f"   - Number of non-null emails: {df['applicant_info.email'].notna().sum()}")
    
    # Check for invalid emails (missing @ symbol)
    invalid_emails = df[~df['applicant_info.email'].str.contains('@', na=False)]['applicant_info.email'].count()
    print(f"   - Emails without '@' symbol: {invalid_emails}")
    
    # Show examples of invalid emails if any
    if invalid_emails > 0:
        print(f"   - Examples: {df[~df['applicant_info.email'].str.contains('@', na=False)]['applicant_info.email'].head(3).tolist()}")

CHECKING SENSITIVE COLUMNS

 Column 'applicant_info.ssn' exists!
   - Number of non-null SSNs: 496
   - Sample values: ['596-64-4340', '425-69-4784', '370-78-5178']

 Column 'applicant_info.ip_address' exists!
   - Number of non-null IPs: 496
   - Sample values: ['192.168.48.155', '10.1.102.112', '10.240.193.250']

 Column 'applicant_info.email' exists (PII)
   - Number of non-null emails: 494
   - Emails without '@' symbol: 1
   - Examples: [nan, 'test.user.outlook.com', nan]
